# `ingestion_WCM.ipynb`
## Wound Care Manual – First Edition (Ministry of Health Malaysia, 2014)

### Document profile

| Property | Detail |
|---|---|
| Total pages | 199 |
| Layout | Single-column (mostly); some pages have figures that fragment text blocks |
| Key tables | Ch14 Modern Dressings (pp 146–149, pdfplumber), Pressure Ulcer Staging (p 108, hardcoded), Analgesics (pp 193–194, pdfplumber) |
| Garbled pages | 16–17, 37, 69, 71, 74, 91, 170–171 — text fragments due to in-page figures; handled with fallback strategy |

### Pages skipped (per scope definition)

| Pages | Reason |
|---|---|
| 1–14 | Cover, TOC, foreword, committee, admin |
| 30–31, 39, 66, 89, 110, 135–136, 144, 158, 164, 183 | Blank / useless separators |
| 50–65 (Ch6) | Wound closure — surgical, not dressing-relevant |
| 40–49 (Ch5) | Nutrition — mostly images (MUST diagram); text summary only kept in Ch9 adjuncts |
| 137–141 (Ch12) | Standard wound dressing SOP — clinical procedure, not RAG-relevant |
| 165–169 (Ch16b) | HBOT — specialised, not first-line dressing |
| 176–180 (Ch17) | Algorithm — identical to GP ingestion; skip to avoid duplication |
| 181–192, 195–199 | Appendices 1–6, 8–10: vitamins, off-loading, footwear, Braden, pain scales, analgesic ladder, morphine protocols |

### Chunk architecture (~32 chunks)

| # | Section | Pages | Source |
|---|---|---|---|
| 1 | Ch1 – Skin Anatomy & Wound Relevance | 15–18 | PyMuPDF text |
| 2 | Ch2 – Wound Definition & Classification | 19–21 | PyMuPDF text |
| 3 | Ch2 – Wound Healing Phases & Factors | 21–25 | PyMuPDF + hardcoded Table 2.1 |
| 4 | Ch3 – Wound Assessment Principles | 26–29 | PyMuPDF text |
| 5 | Ch4 – Wound Infection Pathway & Bacteriology | 32–36 | PyMuPDF text |
| 6 | Ch4 – Antibiotic Treatment for Wound Infection | 37–38 | PyMuPDF text |
| 7 | Ch7a – Burn Wound: Assessment & Management | 67–72 | PyMuPDF text |
| 8 | Ch7b – Traumatic Wound: Assessment & Management | 73–79 | PyMuPDF text |
| 9 | Ch8a – Diabetic Foot Ulcer: Assessment & Classification | 80–85 | PyMuPDF text |
| 10 | Ch8a – Diabetic Foot Ulcer: Management & Foot Care | 85–88 | PyMuPDF text |
| 11 | Ch8b – Venous Ulcer: Classification, Risk & Treatment | 90–95 | PyMuPDF text |
| 12 | Ch8c – Arterial Ulcer: Diagnosis & Management | 96–102 | PyMuPDF text |
| 13 | Ch8d – Pressure Ulcer: Pathophysiology, Staging & Management | 103–109 | PyMuPDF + hardcoded staging |
| 14 | Ch9 – Non-Healing Ulcer: Causes & Management | 111–118 | PyMuPDF text |
| 15 | Ch10 – Life-Threatening Wounds (Necrotizing Fasciitis) | 119–125 | PyMuPDF text |
| 16 | Ch11 – Pain Management in Wound Dressing Procedures | 126–134 | PyMuPDF text |
| 17 | Ch13 – Wound Cleansing Solutions | 142–143 | PyMuPDF text |
| 18 | Ch14 – Dressing Purpose & Categories Overview | 145 | PyMuPDF text |
| 19 | Ch14 – Modern Dressing: Film | 146 | pdfplumber table |
| 20 | Ch14 – Modern Dressing: Hydrogel | 146 | pdfplumber table |
| 21 | Ch14 – Modern Dressing: Hydrocolloid | 147 | pdfplumber table |
| 22 | Ch14 – Modern Dressing: Calcium Alginate | 147 | pdfplumber table |
| 23 | Ch14 – Modern Dressing: Foams | 148 | pdfplumber table |
| 24 | Ch14 – Modern Dressing: Hydrofibre | 148 | pdfplumber table |
| 25 | Ch14 – Modern Dressing: Charcoal | 148 | pdfplumber table |
| 26 | Ch14 – Modern Dressing: Silver | 148 | pdfplumber table |
| 27 | Ch14 – Modern Dressing: Multi-function (Polymeric Membrane) | 149 | pdfplumber table |
| 28 | Ch14 – Modern Dressing: Composite & Other Advanced | 149 | pdfplumber table |
| 29 | Ch15 – Wound Debridement Methods | 150–157 | PyMuPDF text |
| 30 | Ch16a – Honey Dressing: Principles & Application | 159–163 | PyMuPDF text |
| 31 | Ch16c – NPWT: Mechanism, Indications & Settings | 170–175 | PyMuPDF + hardcoded supplement |
| 32 | Appendix 7 – Analgesics Formulations & Dosage | 193–194 | pdfplumber table |


In [8]:
# ── CELL 1 · Imports & configuration ──────────────────────────────────────────
# Uncomment to install if needed:
# !pip install pymupdf pdfplumber -q

import fitz          # PyMuPDF — native PDF text-layer extraction
import pdfplumber    # reliable bordered-table extraction
import re
import json
import hashlib
import unicodedata
from pathlib import Path
from collections import defaultdict

# ── Paths ──────────────────────────────────────────────────────────────────────
PDF_PATH    = "../clinical_pdfs_v2/Wound Care Manual - First Edition.pdf"
SOURCE_NAME = "Wound Care Manual - First Edition.pdf"
OUT_DIR     = Path("../ingestion_output_ai")
OUT_DIR.mkdir(exist_ok=True)

# ── Page index constants (0-based) ────────────────────────────────────────────
# Section A: Basic Wound Principles
PG_CH1_START, PG_CH1_END   = 14, 17   # pp 15–18  Ch1 Skin Anatomy
PG_CH2_START, PG_CH2_END   = 18, 24   # pp 19–25  Ch2 Classification & Healing
PG_CH3_START, PG_CH3_END   = 25, 28   # pp 26–29  Ch3 Wound Assessment
PG_CH4_START, PG_CH4_END   = 31, 37   # pp 32–38  Ch4 Infection

# Section B: Wound Management
PG_CH7A_START, PG_CH7A_END = 66, 71   # pp 67–72  Ch7a Burn
PG_CH7B_START, PG_CH7B_END = 72, 78   # pp 73–79  Ch7b Traumatic
PG_CH8A_START, PG_CH8A_END = 79, 87   # pp 80–88  Ch8a DFU
PG_CH8B_START, PG_CH8B_END = 89, 94   # pp 90–95  Ch8b Venous
PG_CH8C_START, PG_CH8C_END = 95, 101  # pp 96–102 Ch8c Arterial
PG_CH8D_START, PG_CH8D_END = 102, 108 # pp 103–109 Ch8d Pressure
PG_CH9_START,  PG_CH9_END  = 110, 117 # pp 111–118 Ch9 Non-healing
PG_CH10_START, PG_CH10_END = 118, 124 # pp 119–125 Ch10 Life-threatening

# Section C: Practical Wound Care
PG_CH11_START, PG_CH11_END = 125, 133 # pp 126–134 Ch11 Analgesia
PG_CH13_START, PG_CH13_END = 141, 142 # pp 142–143 Ch13 Cleansing
PG_CH14_OVERVIEW           = 144      # p 145 Ch14 overview
PG_CH14_TABLE_START        = 145      # p 146 dressing table start
PG_CH14_TABLE_END          = 148      # p 149 dressing table end
PG_CH15_START, PG_CH15_END = 149, 156 # pp 150–157 Ch15 Debridement
PG_CH16A_START, PG_CH16A_END = 158, 162 # pp 159–163 Ch16a Honey
PG_CH16C_START, PG_CH16C_END = 169, 174 # pp 170–175 Ch16c NPWT
PG_APP7_START,  PG_APP7_END  = 192, 193 # pp 193–194 Appendix 7 Analgesics

# ── Chunking limits ───────────────────────────────────────────────────────────
MAX_CHUNK_CHARS = 3000
MIN_CHUNK_CHARS = 80

print("✅  imports OK")
print(f"   Source: {SOURCE_NAME}")
print(f"   Output dir: {OUT_DIR}")


✅  imports OK
   Source: Wound Care Manual - First Edition.pdf
   Output dir: ..\ingestion_output_ai


In [9]:
# ── CELL 2 · Helper functions ──────────────────────────────────────────────────

# ── Boilerplate patterns specific to WCM ──────────────────────────────────────
# Spaced-out section banner:  "S E C T I O N   A :   B A S I C ... | 12"
SECTION_BANNER_RE = re.compile(
    r'S\s*E\s*C\s*T\s*I\s*O\s*N\s+[A-C]\s*:\s*[A-Z\s]+\|\s*\d+',
    re.IGNORECASE
)
# Chapter header block:  "CHAPTER\n7\n"
# CHAPTER_HEADER_RE = re.compile(r'CHAPTER\s*\n+\s*\d+\s*\n', re.IGNORECASE)
CHAPTER_HEADER_RE = re.compile(r'CHAPTER[\s\n]*\d*[\s\n]*', re.IGNORECASE)
# Standalone page number (1–3 digits on a line by itself)
PAGE_NUM_RE = re.compile(r'^\s*\d{1,3}\s*$', re.MULTILINE)
# References block — strip from "References\n1." to end of text
REFERENCES_RE = re.compile(
    r'\n\s*References?\s*\n\s*1\..*',
    re.DOTALL | re.IGNORECASE
)
# "Point(s) to remember" — keep these (clinically useful); just normalise heading
# Author lines at chapter start: "Dr. FirstName LastName" or "Dr FirstName"
AUTHOR_LINE_RE = re.compile(r'^Dr\.?\s+[A-Z][a-zA-Z\s\.]+$', re.MULTILINE)


def clean_wcm_text(text: str) -> str:
    """
    Full cleaning pipeline for raw WCM page text.
    1. NFKC normalise (ligatures, non-breaking spaces → regular spaces)
    2. Replace \xa0 (non-breaking space) with regular space
    3. Strip spaced-out section banners
    4. Strip standalone page numbers
    5. Strip chapter header blocks
    6. Strip author lines at chapter starts
    7. Strip References sections at end of chapters
    8. Collapse 3+ consecutive blank lines → 2
    """
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\xa0', ' ').replace('\xad', '-')
    text = SECTION_BANNER_RE.sub('', text)
    text = CHAPTER_HEADER_RE.sub('', text)
    text = PAGE_NUM_RE.sub('', text)
    text = AUTHOR_LINE_RE.sub('', text)
    text = REFERENCES_RE.sub('', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def get_pages_text(doc: fitz.Document, start_idx: int, end_idx: int,
                   skip_pages: list = None) -> str:
    """
    Extract and concatenate text from pages [start_idx .. end_idx] (inclusive, 0-based).
    Blocks are sorted by y0 for correct single-column reading order.
    Optionally skip specific page indices (for image-only pages).
    """
    skip_pages = skip_pages or []
    parts = []
    for pg_idx in range(start_idx, end_idx + 1):
        if pg_idx in skip_pages:
            continue
        pg = doc[pg_idx]
        blocks = pg.get_text('blocks', sort=True)
        page_texts = []
        for b in blocks:
            if b[6] != 0:   # skip image blocks
                continue
            t = unicodedata.normalize('NFKC', b[4]).replace('\xa0', ' ').strip()
            if t:
                page_texts.append(t)
        if page_texts:
            parts.append('\n'.join(page_texts))
    raw = '\n\n'.join(parts)
    return clean_wcm_text(raw)


def cell_text(cell) -> str:
    """Normalise a pdfplumber table cell value to a clean single string."""
    if cell is None:
        return ''
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', str(cell))
                  .replace('\xa0', ' ').replace('\xad', '-')).strip()


def make_chunk_id(source: str, section: str, idx: int = 0) -> str:
    raw = f'{source}::{section}::{idx}'
    return hashlib.md5(raw.encode()).hexdigest()[:12]


def make_chunk(section: str, parent_section: str, text: str,
               chunk_index: int = 0) -> dict:
    return {
        'chunk_id':       make_chunk_id(SOURCE_NAME, section, chunk_index),
        'source':         SOURCE_NAME,
        'section':        section,
        'parent_section': parent_section,
        'chunk_index':    chunk_index,
        'char_count':     len(text),
        'text':           text,
        'ai_summary':     text,   # overwrite with LLM summary if ENABLE_AI_SUMMARY = True
    }


def split_long_chunk(text: str, max_chars: int = MAX_CHUNK_CHARS) -> list:
    """Split text at paragraph/sentence boundaries if over max_chars."""
    if len(text) <= max_chars:
        return [text]
    chunks, remaining = [], text
    while len(remaining) > max_chars:
        split_pos = remaining.rfind('\n\n', 0, max_chars)
        if split_pos == -1:
            split_pos = remaining.rfind('. ', 0, max_chars)
        if split_pos == -1:
            split_pos = max_chars
        else:
            split_pos += 2
        chunks.append(remaining[:split_pos].strip())
        remaining = remaining[split_pos:].strip()
    if remaining.strip():
        chunks.append(remaining.strip())
    return [c for c in chunks if len(c) >= MIN_CHUNK_CHARS]


# ── Open PDF ──────────────────────────────────────────────────────────────────
doc = fitz.open(PDF_PATH)
print(f"✅  PDF opened: {doc.page_count} pages")
print(f"   Page size: {doc[0].rect.width:.0f}×{doc[0].rect.height:.0f} pt")


✅  PDF opened: 199 pages
   Page size: 612×792 pt


## Section A · Basic Wound Principles
### Cell 3 — Chapter 1: Skin Anatomy & Wound Relevance (pp 15–18)

In [11]:
# ── CELL 3 · Chapter 1 — Skin Anatomy & Wound Relevance (pp 15–18) ────────────
# Pages 16–17 have figures with fragmented block layout (skin diagrams).
# We skip those image-dominated pages and rely on pp 15, 18 which are clean prose.
# A hardcoded supplement fills the gaps for Dermis/Subcutaneous descriptions.

SKIP_CH1 = [15, 16]   # 0-indexed: pp 16, 17 — figure-heavy, garbled blocks

ch1_text = get_pages_text(doc, PG_CH1_START, PG_CH1_END, skip_pages=SKIP_CH1)

# Supplement: Dermis and Subcutaneous layer descriptions (partially garbled pages)
CH1_SUPPLEMENT = """
1-4. Dermis
The second principal layer of skin. Composed of connective tissue which provides
strength, extensibility and elasticity. Thickness depending on the anatomical site (e.g. very thick in the palms and soles and very thin in the eyelids, penis, and scrotum). Contains nerves, glands, hair follicles and also receptors for heart, cold, pain, pressire, itch and tickle. It has rich blood supply 
from vascular plexus in the deep dermis. Through extensive vascular network (ascending asterioles and capilary loops), the blood supply eventually reaches upper layers of dermis.

1-5. Subcutaneous Layer
The deepest layer of the skin, also known as subcutis or hypodermis. It varies in thickness and depth.
Comprised of adipose tissue, connective tissue and blood vessels. Forms a network of collagen and fat cells.
Responsible for conserving the body's heat and protects body organs from pressure injury.

1-6. Skin and Wound
Superficial wound that damage to the epithelium only, can heal by epithelial regeneration (reconstitute) and
may have little scar formation.
Deeper wound; incisional and excisional skin wounds that damage the dermis will heal through the formation of a collagen scar.
Regeneration requires an intact connective tissue scaffold.
Scar formation occurs if the extracellular matrix framework is damaged, causing alteration of the tissue architecture.
"""

ch1_full = ch1_text + '\n' + CH1_SUPPLEMENT
ch1_full = clean_wcm_text(ch1_full)

print(f"Ch1 text length: {len(ch1_full)} chars")
print(ch1_full[:600])
print("...")
print(ch1_full[-200:])
print(ch1_full)


Ch1 text length: 3140 chars
1 Clinical Applied Anatomy in
Wound Care

1‐1. What is the Skin?
 
Skin is the outer covering of the body and thus provides protection.
 
It is the largest organ in our body in term of weight and surface areas.
 
Its thickness ranges from 0.5 to 4.0 mm depending on location.
 
It consists of different tissues that are joined together to perform several 
essential functions.
 
It is a dynamic organ in a constant of change; whereby the outer layers 
are  continuously  shed  and  replaced  by  the  inner  cells  moving  to  the 
surface.
 
 Structurally, the skin consists of 3 principal lay
...
n of a collagen scar.
Regeneration requires an intact connective tissue scaffold.
Scar formation occurs if the extracellular matrix framework is damaged, causing alteration of the tissue architecture.
1 Clinical Applied Anatomy in
Wound Care

1‐1. What is the Skin?
 
Skin is the outer covering of the body and thus provides protection.
 
It is the largest organ in

### Cell 4 — Chapter 2: Wound Classification, Etiology & Healing Phases (pp 19–25)

In [12]:
# ── CELL 4 · Chapter 2 — Wound Classification & Healing (pp 19–25) ─────────────
# Pages 20 (Figure 2.1 etiology images) and 22 (Figure 2.2 phases graph) are
# predominantly image pages — skip them for text extraction.
# Page 23 (Table 2.1) and pages 21, 24–25 are good prose.

SKIP_CH2 = [19, 21]   # 0-indexed: pp 20 (etiology figure only), 22 (healing graph)

ch2_text = get_pages_text(doc, PG_CH2_START, PG_CH2_END, skip_pages=SKIP_CH2)

# Hardcoded Table 2.1 — Phases of Wound Healing
# (Table is well-structured on p 23 but may render as fragmented columns in fitz)
TABLE_2_1 = """
Table 2.1 — Phases of Wound Healing

Phase         | Timing                 | Key Cells                 | Process Involved                                                                     | Outcome
--------------|------------------------|---------------------------|--------------------------------------------------------------------------------------|-------------------------------------------------------
Haemostasis   | Immediate              | Platelets                 | Vasoconstriction; Platelet adhesion, degranulation & aggregation;                    | Fibrin clot
              |                        |                           | Activation of coagulation cascade                                                    |
Inflammation  | Day 1–3                | Neutrophils, Macrophages  | Vasodilatation; Activation of complement cascade; Infiltration of wound              | Healthy wound bed
              |                        |                           | with neutrophils and monocytes; Phagocytosis of bacteria, foreign body & cell debris |
Proliferation | Day 2 – Week 3         | Fibroblast                | Fibroblast migration; Reconstitution of dermis Fibroplasia & Angiogenesis;           | Granulation tissue; New epithelium; Contracted wound
              |                        |                           | Re-epithelialization; Wound contraction;                                             |
Maturation    | 1 Week – several weeks | None                      | Collagen type III degradation; Collagen type I Synthesis;                            | Increased tissue strength      
"""

ch2_full = ch2_text + '\n' + TABLE_2_1
ch2_full = clean_wcm_text(ch2_full)

print(f"Ch2 text length: {len(ch2_full)} chars")
print(ch2_full[:600])
print("...")
print(ch2_full[-300:])
print(ch2_full)


Ch2 text length: 5281 chars
Definition and Classification of Wound, 
and Stages of Wound Healing
Dr Haris Ali Chemok Ali/ Dr Mohammad Anwar Hau Abdullah
2‐1. Definition
Wound
A  wound  is  an  injury  to  the  integument  or  to  the  underlying 
structures;  
It is visible result of individual cell death or damage; that may or 
may  not  result  in  a  loss  of  skin  integrity  whereby  physiological 
function of the tissue is impaired. 
Ulcer 
An  interruption  of  continuity  of  an  epithelial  surface  with  an 
inflamed base. 
It is usually a result of an underlying or internal etiology.
2‐2. Classification of Wou
...
  |                           | Re-epithelialization; Wound contraction;                                             |
Maturation    | 1 Week – several weeks | None                      | Collagen type III degradation; Collagen type I Synthesis;                            | Increased tissue strength
Definition and Classification of Wound, 
and Stages of Wound Heali

### Cell 5 — Chapter 3: Wound Assessment Principles (pp 26–29)

In [19]:
# ── CELL 5 · Chapter 3 — Wound Assessment (pp 26–29) ─────────────────────────
# Page 27 is the T.I.M.E. figure (mostly image) and page 28 is partially garbled
# (wound documentation examples). We take pages 26 + 29 cleanly, and skip 27–28
# since T.I.M.E. is already fully covered as a dedicated chunk in ingestion_GP.

SKIP_CH3 = [26, 27]   # 0-indexed: pp 27 (T.I.M.E. figure), 28 (documentation examples)

ch3_text = get_pages_text(doc, PG_CH3_START, PG_CH3_END, skip_pages=SKIP_CH3)

# Add brief T.I.M.E. cross-reference note
CH3_NOTE = """
Note: The T.I.M.E. wound bed assessment framework (Tissue, Infection/Inflammation,
Moisture, Epidermal margin) is illustrated in Figure 3.1 of this chapter. A full
structured description of T.I.M.E. is available in the GP Wound Care Guideline chunk
'Wound Assessment — T.I.M.E. Framework'.

Principles of Wound Documentation:
• Timely, Accurate and Objective
• Concise and Comprehensive
• Legible writing; include signature and printed name
• Use only approved abbreviations, organizationally approved abbreviations and colloquialisms
• Regular, Systematic, Standardised, Easily interpreted and Time-efficient
• Used to inform management decisions
• The findings of wound assessment and the dressing solution / material should be documented in wound chart.
"""

ch3_full = ch3_text + '\n' + CH3_NOTE
ch3_full = clean_wcm_text(ch3_full)

print(f"Ch3 text length: {len(ch3_full)} chars")
print(ch3_full[:600])
print(ch3_full)


Ch3 text length: 2075 chars
Wound Assessment and 
Documentation

3‐1. Wound Assessment
General assessment: 
The general assessment is to identify and eliminate any underlying causes 
or contributing factors which may impede the wound healing process; the 
causes include: 
• 
Age (extremes of age ) 
• 
Diseases or co morbidities (e.g. diabetes mellitus , renal impairment )  
• 
Medication (steroids , chemotherapy ) 
• 
Obesity  
• 
Nutrition (refer to on nutrition) 
• 
Impaired blood supply (refer to on arterial and venous ulcers ) 
• 
Lifestyle (smoking , alcohol) 
 
Local wound assessment (Figure 3.1): 
 Local assessmen
Wound Assessment and 
Documentation

3‐1. Wound Assessment
General assessment: 
The general assessment is to identify and eliminate any underlying causes 
or contributing factors which may impede the wound healing process; the 
causes include: 
• 
Age (extremes of age ) 
• 
Diseases or co morbidities (e.g. diabetes mellitus , renal impairment )  
• 
Medication (steroid

### Cell 6 — Chapter 4: Wound Infection & Bacteriology (pp 32–38)

In [20]:
# ── CELL 6 · Chapter 4 — Wound Infection & Bacteriology (pp 32–38) ────────────
# Pages 37–38 (antibiotic treatment + specimen collection) are partially garbled
# due to procedure-box side columns. We extract what we can; key antibiotic
# guidance is readable on p 38.
# We create TWO chunks: (a) infection pathway + bacteriology, (b) antibiotic Tx.

# --- 4a: Infection pathway & bacteriology (pp 32–36) ---
ch4a_text = get_pages_text(doc, PG_CH4_START, PG_CH4_START + 4)   # pp 32–36 (idx 31–35)

# --- 4b: Antibiotic treatment (pp 37–38) ---
ch4b_text = get_pages_text(doc, PG_CH4_START + 5, PG_CH4_END)     # pp 37–38 (idx 36–37)

# Supplement for antibiotic guidance (garbled on p 37; clean summary from p 38)
CH4B_SUPPLEMENT = """
4-5. Antibiotic Treatment
Antibiotics should only be given when there is evidence of wound infection.
Treatment should follow the antibiotic guideline.

Empirical treatment should cover the possible organisms as stated in the table 4-2 above, or follow
the local antimicrobial pattern and should be changed according to the C&S results.

Points to Remember:
• Surface swab has the lowest clinical value
• Sample should be taken when indicated
• Correct sampling technique is pertinent for optimal yield
• Antibiotic treatment only initiated indicated and must follow guideline
"""

ch4a_full = clean_wcm_text(ch4a_text)
ch4b_full  = clean_wcm_text(ch4b_text + '\n' + CH4B_SUPPLEMENT)

print(f"Ch4a (Infection pathway) text length: {len(ch4a_full)} chars")
print(ch4a_full[:400])
print("---")
print(f"Ch4b (Antibiotic Tx) text length: {len(ch4b_full)} chars")
print(ch4b_full[:400])
print(ch4b_full)


Ch4a (Infection pathway) text length: 5025 chars
Wound Infection and Bacteriology    
  in Wound Care
Dr.Nurahan  Maning

4‐1. Pathway of Wound Infection
Exposed wound surfaces provides ideal culture medium for a wide varieties of 
microorganisms  to  contaminate  and  colonized.    Multiplication  of  bacteria 
within a wound can reach a state of ‘critical colonization’ where the bacteria 
will invade viable tissues and leads to infection. 
The
---
Ch4b (Antibiotic Tx) text length: 3275 chars
2. 
Tissue sampling
ould be pe
erformed a
aseptically
y after init
dement an
 
Test request : Tissue for C&S and gram stain 
 
Sho
clea
ansing (wi
ith sterile
saline or d
distilled w
tial debrid
water) of t
he wound
nd 
d
Proce
edure for T
Tissue Sam
mpling
btain tissue
e sample fr
eep part of
ound or bas
se of the le
om the de
esion/wou
aced into a
nd/ulcer a
d containe
the 
nd 
r with
w drops 
2. 
Tissue sampling
ould be pe
erformed a
aseptically
y after init
dement an
 
Test request : Tissu

## Section B · Wound Management
### Cell 7 — Chapter 7a: Burn Wound Management (pp 67–72)

In [21]:
# ── CELL 7 · Chapter 7a — Burn Wound Management (pp 67–72) ────────────────────
# Pages 69–71 are partially garbled (management steps mixed with figure text).
# Key clean content: p 68 (burn depth table), p 70 (referral criteria), p 72
# (indications for referral after conservative management failure).
# We extract all pages and supplement with hardcoded management steps + burn depth table.

ch7a_text = get_pages_text(doc, PG_CH7A_START, PG_CH7A_END)

# Hardcoded burn depth table (clearly structured on page 68)
BURN_DEPTH_TABLE = """
Table 7a.1 — Diagnosis of Burn Depth

Depth               | Epidermal (1st degree) | Partial Thickness – Superficial Dermal (2nd degree)  | Partial Thickness – Deep Dermal (2nd degree) | Full Thickness (3rd degree)
--------------------|------------------------|------------------------------------------------------|----------------------------------------------|----------------------------
Colour              | Hyperemia              | Pale pink                                            | Blotchy red                                  | White leathery
Blisters            | None                   | Present                                              | Present                                      | None
Capillary refill    | Present                | Present                                              | Absent                                       | Absent
Sensation           | Very painful           | Very painful                                         | Absent                                       | Absent
Spontaneous healing | Yes                    | Yes                                                  | No                                           | No
"""

# Hardcoded management summary for garbled pages
BURN_MGMT_SUPPLEMENT = """
7a-2. Management of the Burn Wound

1. Burn injury is a medical emergency. After immediate first aid has been given, the principles ofprimary and secondary
   survey and simultaneous resuscitation should follow as per ATLS principles (Airway, Breathing,
   Circulation, Disability, Exposure, and Fluid Resuscitation).

2. The burn areas assessment using the Lund and Browder chart (Figure 7a.1).
   This will help to indentify patients that need to be referred to a specialized medical facility (see Table 7a-2) 
   and to start fluid resuscitation as per Parkland Formula to estimate the total body surface area (TBSA) burned.

3. Fluid resuscitation: Parkland Formula — 4 ml × body weight (kg) × %TBSA burned
   (Hartmann's solution); half in first 8 hours, half in next 16 hours.

4. Wound care:
   • Clean burns with chlorhexidine solution.
   • For epidermal/superficial partial thickness burns: paraffin gauze, silver
     sulfadiazine (SSD) cream, or modern wound dressings until wound heals.
   • Adequate pain relief (refer to Chapter 11 on analgesia).
   • Referral criteria below must be applied.

5. Dressing change frequency: Daily or as per dressing type.
"""

ch7a_full = clean_wcm_text(ch7a_text + '\n' + BURN_DEPTH_TABLE + '\n' + BURN_MGMT_SUPPLEMENT)

print(f"Ch7a text length: {len(ch7a_full)} chars")
print(ch7a_full[:600])
print("...")
print(ch7a_full[-400:])
print(ch7a_text)

Ch7a text length: 8673 chars
Management of Acute Wound
a) Burn Wound

AIMS: 
1. To be able to perform correct assessment of the depth and extent of
burn injury 
2. To know when to refer to Specialist Hospital  
3. To be able to manage minor burn wounds in a local setting
SCOPE:
For the purpose of this manual, we specifically refer to patients who can be
managed in a non‐specialized centre
7a‐1. Introduction
Burn  injury  is  a  surgical  emergency,  which  requires  prompt  and  aggressive 
treatment. The management of burns requires a multi‐disciplinary approach in 
order to obtain the best aesthetic and functional outco
...
, half in next 16 hours.

4. Wound care:
   • Clean burns with chlorhexidine solution.
   • For epidermal/superficial partial thickness burns: paraffin gauze, silver
     sulfadiazine (SSD) cream, or modern wound dressings until wound heals.
   • Adequate pain relief (refer to on analgesia).
   • Referral criteria below must be applied.

5. Dressing change frequen

### Cell 8 — Chapter 7b: Traumatic Wound Assessment & Management (pp 73–79)

In [22]:
# ── CELL 8 · Chapter 7b — Traumatic Wound (pp 73–79) ─────────────────────────
# Page 74 (treatment principles) is severely fragmented (text around bullet graphics).
# Pages 73, 75–79 are readable.
# Supplement hardcoded treatment principles for page 74.

SKIP_CH7B = [73]   # 0-indexed: p 74 — severely garbled treatment-principles page

ch7b_text = get_pages_text(doc, PG_CH7B_START, PG_CH7B_END, skip_pages=SKIP_CH7B)

CH7B_TREATMENT_SUPPLEMENT = """
7b-2. Treatment Principles

1. Treat life- or limb-threatening conditions first.
2. Initial management follows ATLS principles.
3. Wounds with torrential bleeding or circulatory compromise are managed under ATLS
   primary survey conditions. All other wounds are treated expectantly after settling
   primary conditions.

Management of bleeding:
• Stop bleeding by direct pressure, haemostatic dressings, tourniquet, or artery forceps.

Options for wound closure (depending on assessment):
• Primary closure (sutures, staples, tissue adhesive) — for clean, fresh wounds
• Delayed primary closure (3–5 days) — for contaminated or at-risk wounds
• Secondary intention healing — for heavily contaminated or infected wounds

Local anaesthesia: Depends on age of patient and wound; e.g. lignocaine with or without
adrenaline, nerve block.
"""

ch7b_full = clean_wcm_text(ch7b_text + '\n' + CH7B_TREATMENT_SUPPLEMENT)

print(f"Ch7b text length: {len(ch7b_full)} chars")
print(ch7b_full[:600])
print("...")
print(ch7b_full[-300:])
print(ch7b_full)


Ch7b text length: 5705 chars
Management of Acute Wound
b) Traumatic Wound

AIM:  
To achieve wound healing in the quickest possible manner with minimal 
morbidity and best cosmetic results  
 
SCOPE: 
 Wounds caused by trauma
7b‐1. Clinical Assessment
History 
Main Points:
• 
Time elapsed since injury 
• 
Mechanism of injury 
• 
Cleanliness of wound and possibility of retained foreign body 
• 
AMPLE (Allergy, Medication, Past History, Last Meal, Environment) 
 
Examination 
a. 
Inspection
• 
Anatomical  location  and  possibility  of  associated  more  serious 
injuries (especially neck, chest, abdomen) 
• 
Size wound edg
...
 adhesive) — for clean, fresh wounds
• Delayed primary closure (3–5 days) — for contaminated or at-risk wounds
• Secondary intention healing — for heavily contaminated or infected wounds

Local anaesthesia: Depends on age of patient and wound; e.g. lignocaine with or without
adrenaline, nerve block.
Management of Acute Wound
b) Traumatic Wound

AIM:  
To achieve w

### Cell 9 — Chapter 8a: Diabetic Foot Ulcer (pp 80–88) — Two chunks

In [23]:
# ── CELL 9 · Chapter 8a — Diabetic Foot Ulcer (pp 80–88) ─────────────────────
# Split into two chunks:
#   8a-A: Introduction, pathophysiology, assessment, Wagner classification (pp 80–85)
#   8a-B: Management, treatment algorithm, foot care advice (pp 85–88)

ch8a_A_text = get_pages_text(doc, PG_CH8A_START, PG_CH8A_START + 5)   # pp 80–85 (idx 79–84)
ch8a_B_text = get_pages_text(doc, PG_CH8A_START + 6, PG_CH8A_END)     # pp 86–88 (idx 85–87)

# Hardcoded Wagner Classification (clearly tabular on p 83 but may fragment)
WAGNER_TABLE = """
Wagner Classification of Diabetic Foot Ulcers:

Grade 0: No open lesion; pre- or post-ulcerative lesion. High-risk foot (callus, deformity).
Grade 1: Superficial ulcer involving full skin thickness, not penetrating to subcutaneous tissue.
Grade 2: Deeper ulcer penetrating to tendon, capsule or bone.
Grade 3: Deep ulcer with osteitis, osteomyelitis or abscess.
Grade 4: Localised gangrene — forefoot or heel.
Grade 5: Gangrene of the entire foot requiring major amputation.
"""

ch8a_A_full = clean_wcm_text(ch8a_A_text + '\n' + WAGNER_TABLE)
ch8a_B_full = clean_wcm_text(ch8a_B_text)

print(f"Ch8a-A (Assessment & Classification) length: {len(ch8a_A_full)} chars")
print(f"Ch8a-B (Management & Foot Care) length: {len(ch8a_B_full)} chars")
print("\nCh8a-A preview:")
print(ch8a_A_full[:500])
print("\nCh8a-B preview:")
print(ch8a_B_full[:400])


Ch8a-A (Assessment & Classification) length: 4564 chars
Ch8a-B (Management & Foot Care) length: 3284 chars

Ch8a-A preview:
Management of Chronic Wound
a) Diabetic Foot Ulcer

8a‐1. Introduction
Figure 8a.1 Diabetic foot Ulcer
• 
Diabetic  foot  is  a  foot  that  exhibits  any  pathology  that  results  directly 
from  diabetes  mellitus  or  any  long‐term  (or  "chronic")  complication  of 
diabetes mellitus (Jeffcoate & Harding, 2003).
• 
Diabetic  foot  implies  that  the  pathophysiological  process  of  diabetes 
mellitus  does  something  to  the  foot  that  puts  it  at  increased  risk  for 
“tissue  damag

Ch8a-B preview:
2. Local management:
• 
Wound/ulcer  management:  depending  on  severity  of  wound; 
vascularity and also presence of infection. 
• 
.Debride  infected/necrotic  tissue  follow  by  wound  management 
(refer Wound care Algorithm in ) 
• 
Do not hesitate to perform re‐debridement if indicated.  
• 
Amputation may be the treatment of choice. 
• 
Minimize r

### Cell 10 — Chapter 8b: Venous Ulcer (pp 90–95)

In [24]:
# ── CELL 10 · Chapter 8b — Venous Ulcer (pp 90–95) ───────────────────────────
# Page 91 (risk factors) is garbled due to figure 8b.1.
# We skip page 91 and supplement with hardcoded risk factor list.

SKIP_CH8B = [90]   # 0-indexed: p 91 — garbled risk-factors + figure page

ch8b_text = get_pages_text(doc, PG_CH8B_START, PG_CH8B_END, skip_pages=SKIP_CH8B)

CH8B_RISK_SUPPLEMENT = """
8b-3. Risk Factors and Associated Factors

Risk factors for chronic venous ulcer:
1. Varicose veins
2. Deep vein thrombosis (DVT)
3. Chronic venous insufficiency
4. Poor calf muscle function
5. Obesity
6. History of leg injury
7. Family history

Associated co-morbid factors:
• Diabetes mellitus
• Heart failure
• Hypertension
• Renal disease
• Rheumatoid arthritis
"""

ch8b_full = clean_wcm_text(ch8b_text + '\n' + CH8B_RISK_SUPPLEMENT)

print(f"Ch8b text length: {len(ch8b_full)} chars")
print(ch8b_full[:600])
print("...")
print(ch8b_full[-300:])


Ch8b text length: 4724 chars
Management of Chronic Wound
b)  Venous Ulcer

8b‐1. Introduction
Venous  ulcer  is  the  commonest  cause  of  leg  ulcer  and  it  contributes  to  a 
significant socio‐economic disability in the population as it affects the quality 
of life. It is due to presence of venous hypertension in the lower limbs. 
 
8b‐2. Classification
There are two major groups of venous ulcer, with different treatment options 
and outcomes.
1. 
Ulcer secondary to primary varicose veins
• 
In  this  group  of  patients,  treating  the  varicosities  will  usually 
result in ulcer healing. 
 
2. 
Ulcer secondary to
...
ic venous ulcer:
1. Varicose veins
2. Deep vein thrombosis (DVT)
3. Chronic venous insufficiency
4. Poor calf muscle function
5. Obesity
6. History of leg injury
7. Family history

Associated co-morbid factors:
• Diabetes mellitus
• Heart failure
• Hypertension
• Renal disease
• Rheumatoid arthritis


### Cell 11 — Chapter 8c: Arterial Ulcer (pp 96–102)

In [25]:
# ── CELL 11 · Chapter 8c — Arterial Ulcer (pp 96–102) ────────────────────────
# Pages 97 is partially garbled (clinical examination bulleted list around figure).
# Key content on pp 96, 98–102 is readable.

SKIP_CH8C = [96]   # 0-indexed: p 97 — clinical exam list partially garbled

ch8c_text = get_pages_text(doc, PG_CH8C_START, PG_CH8C_END, skip_pages=SKIP_CH8C)

CH8C_EXAM_SUPPLEMENT = """
8c-3. Diagnosis

Symptoms of chronic limb ischaemia:
• Intermittent claudication (pain on walking, relieved by rest)
• Rest pain (persistent, worse at night)

Clinical Examination:
• Inspection: Pale, hairless, thin skin; dry ulcer without inflammatory signs;
  Sites — toes, heel, lateral malleolus, dorsum of foot
• Signs of chronic ischaemia: Muscle atrophy, hair loss, skin changes
• Examination: Digital palpation of peripheral pulses
  - Two plus (++) palpable pulses = normal
  - One plus (+) = reduced
  - Negative = absent; arterial disease confirmed

Investigations:
• ABSI (Ankle-Brachial Systolic Index):
  - Normal ≥ 0.9  |  Mild-moderate: 0.5–0.9  |  Severe: < 0.5  |  Critical: < 0.3
• Duplex ultrasound / Angiography for revascularisation planning
"""

# Hardcoded amputee rehabilitation phases (Table 8c.1, p 100)
TABLE_8C1 = """
Table 8c.1 — Phases of Amputee Rehabilitation

1. Pre-operative:   Medical & body assessment, patient education, surgical-level discussion,
                    functional expectations, phantom limb discussion
2. Amputation/dressing: Residual-limb length, myoplastic closure, soft-tissue coverage,
                    nerve handling, rigid dressing, limb reconstruction
3. Acute post-surgical: Wound healing, pain control, proximal body motion, emotional support,
                    phantom limb discussion
4. Pre-prosthetic:  Residual-limb shaping/shrinking, muscle strength, figure-of-8 stump bandaging
5. Prosthetic prescription: Depends on cognitive, medical, functional status and social factors
"""

ch8c_full = clean_wcm_text(ch8c_text + '\n' + CH8C_EXAM_SUPPLEMENT + '\n' + TABLE_8C1)

print(f"Ch8c text length: {len(ch8c_full)} chars")
print(ch8c_full[:600])


Ch8c text length: 7161 chars
Management of Chronic Wound
c)   Arterial Ulcer
Dr Hanif Hussein/ Dr Khairiah Mohd Yatim

8c‐1. Introduction
Arterial ulcers are ischemic ulcers in patients with peripheral vascular disease. 
Reduced blood supply to the affected limb impedes healing and causes delay 
or  non‐healing  of  the  ulcer.  It  is  crucial  to  identify  arterial  ulcers,  as  the 
management would involve revascularization to improve the circulation of the 
affected limb to achieve wound healing.
8c‐2. Risk Factors
Risk factors for chronic limb ischemia include:
1. 
Diabetes mellitus 
2. 
Smoking 
3. 
Dyslipidemia 



### Cell 12 — Chapter 8d: Pressure Ulcer — Pathophysiology, Staging & Management (pp 103–109)

In [26]:
# ── CELL 12 · Chapter 8d — Pressure Ulcer (pp 103–109) ──────────────────────
# Table 8d.1 (Staging, p 108) is detected by pdfplumber as 2 columns only;
# stage descriptions are readable from fitz. We use hardcoded staging data
# which is cleaner than the raw extraction.

ch8d_text = get_pages_text(doc, PG_CH8D_START, PG_CH8D_END)

# Hardcoded pressure ulcer staging (NPUAP 2007, from pp 108–109)
PRESSURE_STAGING = """
Table 8d.1 — Staging of Pressure Ulcer (NPUAP 2007)

Stage I:
  Intact skin with non-blanchable redness of a localised area usually over a bony
  prominence. Darkly pigmented skin may not show visible blanching; colour may differ
  from surrounding area.

Stage II:
  Partial thickness loss of dermis presenting as a shallow open ulcer with a red/pink
  wound bed, without slough. May also present as an intact or open/ruptured serum-filled
  blister.

Stage III:
  Full thickness tissue loss. Subcutaneous fat may be visible but bone, tendon or muscle
  is NOT exposed. Slough may be present but does not obscure the depth of tissue loss.
  May include undermining and tunnelling.

Stage IV:
  Full thickness tissue loss with exposed bone, tendon or muscle. Slough or eschar may
  be present. Often includes undermining and tunnelling.

Suspected Deep Tissue Injury (DTI):
  Purple or maroon localised area of intact skin, or blood-filled blister, caused by
  underlying soft tissue damage from pressure and/or shear.

Unstageable:
  Full thickness tissue loss; depth unknown because wound base is covered by slough
  (yellow, tan, grey, green or brown) or eschar (tan, brown or black).
"""

# Prevention and management supplement
PRESSURE_MGMT = """
8d-3. Management

Prevention is paramount:
• Proper bed positioning; turn patient every 2 hours.
• Check bony prominences regularly.
• Use appropriate padding and pressure-redistributing mattresses/cushions.
• Moisture management: keep skin clean and dry; moisture barrier creams.

General treatment:
• Restoration of tissue perfusion by relief of pressure.
• Treat underlying systemic problems (anaemia, hypoproteinaemia, diabetes).
• Prevent or treat infection (refer to national antibiotic guidelines).
• Improve general health and nutrition.
• For local wound management: follow Wound Care Algorithm (T.I.M.E. principle applies).
  - Necrotic/sloughy pressure ulcer: debridement first (autolytic, enzymatic or surgical).
  - Clean, granulating pressure ulcer: moisture-retentive dressings (foam, hydrocolloid).
  - Infected pressure ulcer: antimicrobial dressings (silver, iodine) + systemic antibiotics.
"""

ch8d_full = clean_wcm_text(ch8d_text + '\n' + PRESSURE_STAGING + '\n' + PRESSURE_MGMT)

print(f"Ch8d text length: {len(ch8d_full)} chars")
print(ch8d_full[:600])


Ch8d text length: 8135 chars
Management of Chronic Wound
d) Pressure Ulcer
Dr Khairiah Mohd Yatim/ Dr Zairizam Zakaria

8d‐1. Definition
A pressure ulcer is localized injury to the skin and/or underlying tissue usually 
over a bony prominence, as a result of pressure, or pressure in combination 
with  shear  and/or  friction.  A  number  of  contributing  or  confounding  factors 
are also associated with pressure ulcers; the significance of these factors is yet 
to be elucidated. (NPUAP 2007)
8d‐2. Pathophysiology
I)  
Primary Factors
• 
Pressure
Kosiak 1961 Arch Phys Meds & Rehab (Animal Study)
– 
There  is  an  inverse


### Cell 13 — Chapter 9: Non-Healing Ulcer (pp 111–118)

In [27]:
# ── CELL 13 · Chapter 9 — Non-Healing Ulcer (pp 111–118) ─────────────────────
# Pages 111–118 are mostly clean single-column text.
# The algorithm figure on p 117 (Figure 9.1) has text blocks readable via fitz.

ch9_full = get_pages_text(doc, PG_CH9_START, PG_CH9_END)
ch9_full = clean_wcm_text(ch9_full)

print(f"Ch9 text length: {len(ch9_full)} chars")
print(ch9_full[:600])
print("...")
print(ch9_full[-300:])


Ch9 text length: 5080 chars
Management of 
Non‐Healing Ulcer

9‐1. Definition
Any wound that has no signs of healing process within 2‐4weeks after 
intervention by proper wound management team. 
 
Features of non‐healing wound: 
• 
Size remain the same
• 
Persistent discharge
Arbitrary guide only (use personal clinical judgement to decide)
9‐2. Causes
Usually multifactorial but can be broadly classified into the following factors
1. 
Patient/Systemic:
• 
Malnutrition
• 
Poorly controlled comorbid conditions e.g Diabetes
• 
Smoking
• 
Hygiene
• 
Medication e.g Steroids/Chemotherapy
• 
Pressure wounds
• 
Patient immobility
...
m for treating non‐healing wounds

Point to Remember: 
• You should recognize when a wound can be termed  non‐healing and may 
need  specialised intervention.  
 
• Palliative care may be appropriate in certain situations.  The possibility of 
malignant change in such ulcers must also be remembered.


### Cell 14 — Chapter 10: Life-Threatening Wounds – Necrotizing Fasciitis (pp 119–125)

In [28]:
# ── CELL 14 · Chapter 10 — Life-Threatening Wounds (pp 119–125) ──────────────
# Page 123 is mostly clinical photos (Figure 10.1); skip it for text extraction.
# Page 124 (Figure 10.2 management flowchart) has some readable text.

SKIP_CH10 = [122]   # 0-indexed: p 123 — clinical photos page

ch10_full = get_pages_text(doc, PG_CH10_START, PG_CH10_END, skip_pages=SKIP_CH10)
ch10_full = clean_wcm_text(ch10_full)

print(f"Ch10 text length: {len(ch10_full)} chars")
print(ch10_full[:600])
print("...")
print(ch10_full[-300:])


Ch10 text length: 4906 chars
Management of 
Life‐Threatening Wounds

10‐1. Introduction
Necrotizing fasciitis (NF):
• 
Necrotizing fasciitis is potentially fatal infection characterized by rapid 
progression with widespread necrosis of the subcutaneous tissue and 
superficial fascia. The infection may also leads to gas production and 
sepsis. It is more likely to occur in immune compromised people.
• 
Subtype of NF and the causative organism:
Flash eating 
bacteria
Subtype 
Bacteriology 
Known as  
Type I 
Polymicrobial
 
Type II 
Group A beta hemolytic Streptococcal
(Streptococcus pyogenes), 
Staphylococcus aureus
Gas ga
...
e, and wound bed 
preparation for secondary wound 
closure 
 
 
Figure 10.2 Flow chart in management of life‐threatening wound

Point to Remember:
• Necrotising fasciitis is a potentially life threatening infection. 
• Early diagnosis and prompt treatment is the most crucial for 
patient’s prognosis


## Section C · Practical Aspects in Wound Care
### Cell 15 — Chapter 11: Pain Management in Wound Dressing Procedures (pp 126–134)

In [29]:
# ── CELL 15 · Chapter 11 — Analgesia for Wound Dressing (pp 126–134) ─────────
# This chapter covers pain types, assessment, and pharmacological management
# protocols. Relevant for clinical staff managing dressing changes.
# Pages 126–134 are mostly clean prose.

ch11_full = get_pages_text(doc, PG_CH11_START, PG_CH11_END)
ch11_full = clean_wcm_text(ch11_full)

# This section may be long — split if needed
ch11_parts = split_long_chunk(ch11_full)
print(f"Ch11 text length: {len(ch11_full)} chars  →  {len(ch11_parts)} sub-chunks")
print(ch11_full[:600])
print("...")
print(ch11_full[-300:])


Ch11 text length: 11180 chars  →  5 sub-chunks
Analgesia for Wound 
Dressing Related Procedures
Dr Kavita Bhojwani/ Dr Mary Cardosa/ Dr Harijah Wahidin

11‐1. Introduction
‘Pain is an important aspect of wound care. “Unresolved pain negatively 
affects wound healing and has an impact on the quality of life. Pain at wound 
dressing procedures can be managed by a combination of accurate 
assessment, suitable dressing choices, skilled wound management and 
individualized analgesic regimens. For therapeutic as well as humanitarian 
reasons, it is vital that clinicians know how to assess, evaluate and manage 
pain (ref 2)’
11‐2. Types of pain
1
...
 a regular basis and then as the wound heals, 
it can be on a PRN basis. 
 
It should be emphasized that prior to any procedure that they may require, 
they will need to ingest their analgesic 1 hour before the procedure. This may 
mean they take the medication at home before arriving at the clinic.


### Cell 16 — Chapter 13: Wound Cleansing Solutions (pp 142–143)

In [30]:
# ── CELL 16 · Chapter 13 — Wound Cleansing (pp 142–143) ──────────────────────
# Clean two-page chapter; fully readable.

ch13_full = get_pages_text(doc, PG_CH13_START, PG_CH13_END)
ch13_full = clean_wcm_text(ch13_full)

print(f"Ch13 text length: {len(ch13_full)} chars")
print(ch13_full)


Ch13 text length: 2197 chars
Wound Cleansing

13-1. Definition
Wound  cleansing  is  a  process  of  removing  inflammatory  contaminants  from 
the wound surface. These contaminants can impede healing and increase risk 
of infection.
The contaminants are:
1. 
Necrotic tissues 
2. 
Excess exudates 
3. 
Foreign objects 
4. 
Infected tissues
Solutions used in wound cleansing can be either non‐antiseptic or antiseptic.
13‐2. Non antiseptic solutions
Non‐antiseptic  solutions  are  used  to  clean  wounds.  Commonly  used  non‐
antiseptic solutions are:
1. 
Normal saline 
• 
Preferred cleanser for most types of wounds (physiologic and safe) 
• 
Less effective in dirty and necrotic wounds 
• 
Not advisable  in MRSA and Pseudomonas infected wound  
• 
Once the container is opened, it should be used within 24 hours 
 
2. 
Water for irrigation 
• 
Less  physiologic  compared  to  normal  saline  but  still  safe  to  be 
used 
• 
Can be used in MRSA and Pseudomonas infected wound
 2 9

13-3.  

### Cell 17 — Chapter 14: Dressing Purpose & Categories Overview (p 145)

In [31]:
# ── CELL 17 · Chapter 14 — Dressing Purpose & Categories Overview (p 145) ─────
# Page 145 contains dressing purpose bullet list and categories (conventional vs modern).
# This serves as a standalone overview chunk.

ch14_overview_full = get_pages_text(doc, PG_CH14_OVERVIEW, PG_CH14_OVERVIEW)
ch14_overview_full = clean_wcm_text(ch14_overview_full)

print(f"Ch14 overview length: {len(ch14_overview_full)} chars")
print(ch14_overview_full)


Ch14 overview length: 790 chars
Types of Dressing

14-1. Dressing Purpose
• 
Protect wound from trauma and microbial contamination  
• 
Reduce pain  
• 
Maintain temperature & moisture of wound 
• 
Absorb drainage & debride the wound 
• 
Control & Prevent haemorrhage (pressure dressing) 
• 
Provide psychological comfort
Ideal/optimum dressing
• 
Remove excess exudates    
 
 
 
 
• 
Waterproof 
• 
Maintain moist wound healing environment   
 
• 
Trauma protection 
• 
Allows gaseous exchange if appropriate 
 
 
• 
Non adherent 
• 
Provide barrier to pathogens 
 
 
 
 
• 
Safe & easy to use 
• 
Provide thermal insulation
14-2. Dressing Categories
1. 
Conventional/ Inert
2. 
Modern/ Advanced/ Active Dressings
Conventional/ Inert
• 
Gauze –soaked with normal saline / antiseptic 
• 
Gamgee as secondary dressing
 3 2


### Cell 18 — Chapter 14: Modern Dressings Table — pdfplumber extraction (pp 146–149)

In [32]:
# ── CELL 18 · Chapter 14 — Modern Dressings Table (pdfplumber, pp 146–149) ────
#
# Layout: 5-column bordered table across 4 pages.
# Columns: DRESSINGS | PURPOSE | ADVANTAGES | DISADVANTAGES | PRACTICAL USAGE
# pdfplumber detects 7 columns due to header spanning; actual data is in cols 0-4.
#   Col 0: Dressing name/number
#   Col 1: Purpose
#   Col 2: Advantages
#   Col 3: Disadvantages
#   Col 4: Practical Usage (frequency of change + application notes)
#
# Strategy: extract all rows from pp 146–149, skip header rows, one chunk per dressing.

DRESSING_PAGES = list(range(PG_CH14_TABLE_START, PG_CH14_TABLE_END + 1))   # 145–148 (0-indexed)

def is_table_header_row(row: list) -> bool:
    """Identify and skip repeated column-header rows."""
    first = cell_text(row[0]) if row else ''
    second = cell_text(row[1]) if len(row) > 1 else ''
    return (first.upper() in ('DRESSINGS', '') and
            second.upper() in ('PURPOSE', 'ADVANTAGES', ''))


def parse_dressing_rows(rows: list) -> dict:
    """
    Group pdfplumber rows into per-dressing dictionaries.
    A new dressing starts whenever col 0 is non-empty (dressing name + number).
    Returns ordered dict: {dressing_name: {purpose, advantages, disadvantages, practical}}
    """
    dressings = {}
    current = None

    for row in rows:
        if is_table_header_row(row):
            continue
        name_raw = cell_text(row[0]) if row else ''
        purpose  = cell_text(row[1]) if len(row) > 1 else ''
        adv      = cell_text(row[2]) if len(row) > 2 else ''
        disadv   = cell_text(row[3]) if len(row) > 3 else ''
        # Practical usage is in col 4 (despite header being in col 5)
        practical = cell_text(row[4]) if len(row) > 4 else ''
        # Also try col 5 in case pdfplumber shifts it
        if not practical and len(row) > 5:
            practical = cell_text(row[5])

        if name_raw:
            current = name_raw
            dressings[current] = {
                'name':       current,
                'purpose':    purpose,
                'advantages': adv,
                'disadvantages': disadv,
                'practical':  practical,
            }
        elif current and any([purpose, adv, disadv, practical]):
            # Continuation row — append to current dressing
            for field, val in [('purpose', purpose), ('advantages', adv),
                                ('disadvantages', disadv), ('practical', practical)]:
                if val:
                    dressings[current][field] = (dressings[current][field] + ' ' + val).strip()

    return dressings


def format_dressing_chunk(d: dict) -> str:
    """Format one dressing entry as a self-contained RAG chunk."""
    lines = [
        f"WCM Modern/Advanced Dressing — {d['name']}",
        f"",
        f"Purpose: {d['purpose']}",
        f"Advantages: {d['advantages']}",
        f"Disadvantages: {d['disadvantages']}",
        f"Practical Usage / Application: {d['practical']}",
        f"",
        f"(Source: Wound Care Manual, First Edition 2014, MOH Malaysia, Chapter 14, pp 146–149)",
    ]
    return '\n'.join(lines)


# ── Extract tables from all four dressing pages ───────────────────────────────
all_dressing_rows = []
with pdfplumber.open(PDF_PATH) as pdf:
    for pg_idx in DRESSING_PAGES:
        page = pdf.pages[pg_idx]
        for tbl in page.find_tables():
            rows = tbl.extract()
            all_dressing_rows.extend(rows)

print(f"Total raw dressing rows extracted: {len(all_dressing_rows)}")

# ── Parse into per-dressing dicts ─────────────────────────────────────────────
dressing_dict = parse_dressing_rows(all_dressing_rows)

print(f"\nDressings found: {len(dressing_dict)}")
for name, d in dressing_dict.items():
    purpose_preview = d['purpose'][:60]
    print(f"  {name!r:50s}  purpose: {purpose_preview!r}")


Total raw dressing rows extracted: 23

Dressings found: 11
  '1. Film'                                           purpose: 'Protect against contamination and friction Maintain moist su'
  '2. Hydrogel'                                       purpose: 'Rehydrate , debride and deslough the wound Promote moist hea'
  '3. Hydro‐ colloid'                                 purpose: 'Provide moist environment Absorb exudates Bacterial barrier'
  '4. Calcium Alginate'                               purpose: 'Absorb wound exudates and maintain moisture'
  '5. Foams'                                          purpose: 'Absorbent Cushioning'
  '6. Hydrofibre'                                     purpose: 'Manage heavy exuding wounds Maintains moist healing environm'
  '7. Charcoal'                                       purpose: 'Odour absorbent'
  '8. Silver'                                         purpose: 'To reduce bacterial bioburden in infected wounds'
  '9. Multi‐ function dressing (Polymeric membra

In [33]:
# ── CELL 19 · Preview dressing chunks ─────────────────────────────────────────
# Verify each dressing chunk looks complete and correct.

dressing_chunks = {}
for name, d in dressing_dict.items():
    text = format_dressing_chunk(d)
    dressing_chunks[name] = text

# Spot-check Film and Silver
for check_name in list(dressing_dict.keys())[:3]:
    print(f"\n{'='*60}")
    print(dressing_chunks[check_name])



WCM Modern/Advanced Dressing — 1. Film

Purpose: Protect against contamination and friction Maintain moist surface Prevent evaporation Facilitate assessment
Advantages: Adherent Transparent with measurement grid Bacterial barrier Waterproof Breathable
Disadvantages: Fluid collection Possibility of stripping away newly formed epithelium on removal
Practical Usage / Application: Apply the film over the site making sure there is no air under it To remove the film, stretch the film and pull slowly from the edges Frequency of dressing change: 2‐5 days depending on the wound

(Source: Wound Care Manual, First Edition 2014, MOH Malaysia, Chapter 14, pp 146–149)

WCM Modern/Advanced Dressing — 2. Hydrogel

Purpose: Rehydrate , debride and deslough the wound Promote moist healing Cavity filling
Advantages: Comfortable Provide moist environment and reduce pain Rehydrate eschar Desloughing agent Promotes granulation
Disadvantages: Need secondary dressing Maceration of the skin around the wound
P

### Cell 20 — Chapter 15: Wound Debridement Methods (pp 150–157)

In [34]:
# ── CELL 20 · Chapter 15 — Wound Debridement (pp 150–157) ────────────────────
# Pages 150–157 are mostly clean prose. Some pages have small instrument photos
# with captions but text content is intact.

ch15_full = get_pages_text(doc, PG_CH15_START, PG_CH15_END)
ch15_full = clean_wcm_text(ch15_full)

# May be long — split if needed
ch15_parts = split_long_chunk(ch15_full)
print(f"Ch15 text length: {len(ch15_full)} chars  →  {len(ch15_parts)} sub-chunk(s)")
print(ch15_full[:600])
print("...")
print(ch15_full[-300:])


Ch15 text length: 7483 chars  →  3 sub-chunk(s)
Wound Debridement
Dr.Mohamed Yusof Hj Abdul Wahab  / Dr.Harikrishna K.R Nair/

Dr.Zairudin Abdullah Zawawi
15-1. Introduction
It is a process of removal of non viable tissue and contaminants from a wound 
to promote healing.
Methods of debridement:
• 
Surgical
• 
Autolytic
• 
Enzymatic
• 
Mechanical
• 
Biological
• 
Hydrostatic 
 
15-2. Surgical Debridement
Removal  of  necrotic  tissue  by  sharp  debridement  using  a  scalpel,  scissors, 
curette, Humby knife, electrical dermatome or other instrument to cut necrotic 
tissue from a wound  as to prepare the wound bed for optimum healing.
Indi
...
ostatic debridement is more precise and more selective than a 
scalpel 
• 
Enable surgeon to precisely target (through different power settings) and 
removes devitalised tissue and contaminants and at the same time 
preserves collateral healthy tissue
Figure 15.4 High Power Hydrostatic Debrider
 4 4


### Cell 21 — Chapter 16a: Honey Dressing — Principles & Application (pp 159–163)

In [35]:
# ── CELL 21 · Chapter 16a — Honey Dressing (pp 159–163) ──────────────────────
# Pages 159–163: introduction and properties (pp 159–160), application table
# (pp 161–162), and points to remember (p 163).
# The honey dressing procedure table (pp 161–162) is extractable via pdfplumber.

ch16a_prose = get_pages_text(doc, PG_CH16A_START, PG_CH16A_END)
ch16a_prose = clean_wcm_text(ch16a_prose)

# Extract honey procedure table via pdfplumber
HONEY_TABLE_PAGES = [160, 161]   # 0-indexed: pp 161, 162
honey_steps = []
with pdfplumber.open(PDF_PATH) as pdf:
    for pg_idx in HONEY_TABLE_PAGES:
        page = pdf.pages[pg_idx]
        for tbl in page.find_tables():
            rows = tbl.extract()
            for row in rows:
                step_num = cell_text(row[0]) if row else ''
                step_txt = cell_text(row[1]) if len(row) > 1 else ''
                if step_num.strip().isdigit() and step_txt:
                    honey_steps.append(f"Step {step_num}: {step_txt}")

HONEY_PROCEDURE = "\nHoney Dressing Application Procedure:\n" + "\n".join(honey_steps) if honey_steps else ''

ch16a_full = clean_wcm_text(ch16a_prose + '\n' + HONEY_PROCEDURE)

print(f"Ch16a text length: {len(ch16a_full)} chars")
print(ch16a_full[:600])
print("...")
print(ch16a_full[-300:])


Ch16a text length: 5521 chars
Adjunctive Teatment
a)  Honey Dressing

16a-1. Introduction
1. Honey has been rediscovered to have medicinal value in treating wounds. 
Numerous  studies  have  compared  honey  with  other  modern  dressings  in 
managing various type of wounds.
2. Honey is mainly used to promote granulation and epithelization of a wound 
for  secondary  intention  healing  or  to  be  followed  by  further  surgical 
procedure  for  soft  tissue  coverage,  e.g:  split  skin  graft,  full  thickness  skin 
graft, flaps.
3. Honey has antibacterial effects which are attributed to its high osmolarity, 
low pH, 
...
rs.
 4 9

Point to Remember:
• Although it has debridement activity, its main functions are to 
promote granulation and epithelization of a wound.
• Any type of honey can be used, but the most suitable is the 
therapeutic honey.
• Honey dressing is not a substitute for a proper surgical debridement.


### Cell 22 — Chapter 16c: NPWT — Mechanism, Indications, Contraindications & Settings (pp 170–175)

In [36]:
# ── CELL 22 · Chapter 16c — NPWT (pp 170–175) ────────────────────────────────
# Pages 170–171 are severely garbled (fragmented heading + mechanism text around figures).
# Pages 172–175 are mostly clean.
# Supplement pages 170–171 with hardcoded content.

SKIP_CH16C = [169, 170]   # 0-indexed: pp 170, 171 — severely garbled

ch16c_text = get_pages_text(doc, PG_CH16C_START, PG_CH16C_END, skip_pages=SKIP_CH16C)

# Hardcoded supplement for garbled pages 170–171 (introduction + mechanism)
CH16C_SUPPLEMENT = """
Chapter 16c — Negative Pressure Wound Therapy (NPWT)

16c-1. Introduction
NPWT provides a new paradigm in wound care, overcoming the limitations of standard
existing techniques and achieving a high rate of success across multiple clinical
disciplines.

16c-2. What is NPWT?
NPWT is a recent technique that applies subatmospheric (negative) pressure to the wound
via a specialised open-cell wound dressing (e.g. sponge/gauze) placed directly on the
wound, covered with an occlusive film. Negative pressure is delivered across the entire
wound surface by a vacuum pump.

Components of NPWT dressing:
a. Sterile open-cell wound interface (foam or gauze)
b. Pliable vacuum tubing
c. An occlusive adhesive film (airtight seal)
d. Canister (collection chamber)
e. Vacuum pump

16c-3. How NPWT Works
1. Provides a closed, moist wound healing environment; encourages granulation tissue
   growth at wound site; reduces contamination from external bacteria.
2. Decreases wound volume; draws wound edges together, approximating the wound.
3. Removes excess fluids (exudate); helps decrease bacterial colonisation at wound site.
4. Helps remove interstitial fluid; reduces oedema; improves blood flow to the wound.
5. Promotes granulation tissue growth through the mechanical force of suction pressure
   (stimulates cell mitosis by micro-deformation of wound surface).
"""

ch16c_full = clean_wcm_text(CH16C_SUPPLEMENT + '\n' + ch16c_text)

print(f"Ch16c text length: {len(ch16c_full)} chars")
print(ch16c_full[:600])
print("...")
print(ch16c_full[-400:])


Ch16c text length: 4258 chars
c — Negative Pressure Wound Therapy (NPWT)

16c-1. Introduction
NPWT provides a new paradigm in wound care, overcoming the limitations of standard
existing techniques and achieving a high rate of success across multiple clinical
disciplines.

16c-2. What is NPWT?
NPWT is a recent technique that applies subatmospheric (negative) pressure to the wound
via a specialised open-cell wound dressing (e.g. sponge/gauze) placed directly on the
wound, covered with an occlusive film. Negative pressure is delivered across the entire
wound surface by a vacuum pump.

Components of NPWT dressing:
a. Sterile o
...
ration and pressure damage to skin areas adjacent to the wound. 
4. 
Reduction in perfusion caused by pressure on small caliber vessels.
 6 1

Points to remember:  
• NPWT  is  only  an  adjunct  to  the  management  of  chronic,  acute  and 
difficult wounds and it is not a panacea. 
• NPWT prepares wound bed for a greater chance of successful closure. 
• NPWT d

### Cell 23 — Appendix 7: Analgesics Formulations & Dosage (pp 193–194) — pdfplumber

In [37]:
# ── CELL 23 · Appendix 7 — Analgesics Formulations & Dosage (pdfplumber) ──────
# Two-page table of commonly used analgesics (Paracetamol, NSAIDs, Weak and
# Strong Opioids). pdfplumber detects the bordered table well on both pages.

APP7_PAGES = list(range(PG_APP7_START, PG_APP7_END + 1))   # [192, 193] (0-indexed)

analgesic_rows = []
with pdfplumber.open(PDF_PATH) as pdf:
    for pg_idx in APP7_PAGES:
        page = pdf.pages[pg_idx]
        for tbl in page.find_tables():
            rows = tbl.extract()
            analgesic_rows.extend(rows)

print(f"Analgesic table rows: {len(analgesic_rows)}")

# Format the table as clean structured text
def format_analgesic_table(rows: list) -> str:
    """
    Reconstruct the analgesics table as readable key-value text.
    Columns: DRUG | FORMULATION AVAILABLE | DOSAGE
    Handle multi-row drug entries (e.g. Morphine spans several rows).
    """
    lines = [
        "Appendix 7 — Formulations and Dosage of Commonly Used Analgesics",
        "(Source: Wound Care Manual, First Edition 2014, MOH Malaysia)",
        "",
    ]
    current_drug = None
    current_formulation = []
    current_dosage = []

    def flush_drug():
        if current_drug:
            lines.append(f"Drug: {current_drug}")
            if current_formulation:
                lines.append(f"  Formulations: {' | '.join(current_formulation)}")
            if current_dosage:
                lines.append(f"  Dosage: {' | '.join(current_dosage)}")
            lines.append("")

    for row in rows:
        drug_raw = cell_text(row[0]) if row else ''
        form_raw = cell_text(row[1]) if len(row) > 1 else ''
        dose_raw = cell_text(row[2]) if len(row) > 2 else ''

        # Skip pure header rows
        if drug_raw.upper() in ('DRUG', 'FORMULATION', 'AVAILABLE', ''):
            if form_raw.upper() in ('FORMULATION', 'AVAILABLE', ''):
                continue

        # Detect section headers (e.g. "NSAIDs", "WEAK OPIOIDs", "STRONG OPIOIDs")
        if drug_raw and not form_raw and not dose_raw:
            flush_drug()
            current_drug = None
            lines.append(f"--- {drug_raw} ---")
            current_formulation = []
            current_dosage = []
            continue

        if drug_raw:
            # New drug entry
            flush_drug()
            current_drug = drug_raw
            current_formulation = [form_raw] if form_raw else []
            current_dosage = [dose_raw] if dose_raw else []
        else:
            # Continuation of current drug
            if form_raw:
                current_formulation.append(form_raw)
            if dose_raw:
                current_dosage.append(dose_raw)

    flush_drug()
    return '\n'.join(lines)


app7_text = format_analgesic_table(analgesic_rows)
print(app7_text[:1000])


Analgesic table rows: 68
Appendix 7 — Formulations and Dosage of Commonly Used Analgesics
(Source: Wound Care Manual, First Edition 2014, MOH Malaysia)

Drug: Paracetamol
  Formulations: Tablet 500mg, | Suspension 500mg/5ml, | Suppositories
  Dosage: 500 mg – 1gm qid

--- NSAIDs ---
Drug: Diclofenac
  Formulations: Tablet 50mg & 25mg, | IM injections | Suppositories 12.5mg, | 25mg, (50mg & 100mg)* | Gel
  Dosage: Oral 25 ‐ 50mg tds, (max 3 | doses/day) | IM 25‐50mgs tds ( max | 3doses/day) (not encouraged) | Sup: 50mg‐100mg stat

Drug: Mefenamic Acid
  Formulations: Capsule 250mg
  Dosage: 250 mg – 500mg tds

--- (Ponstan) ---
Drug: Ibuprofen ( Brufen)
  Formulations: Tablet 200mg & 400mg*
  Dosage: 200 mg – 400 mg tds

Drug: Naproxen
  Formulations: Tablet 250mg, 550mg
  Dosage: 500mg‐550 mg bd

--- (Naprosyn, Synflex) ---
Drug: Ketoprofen
  Formulations: Capsule 100mg *,
  Dosage: Oral: 100mg daily, IV: 100mg bd

Drug: (Orudis, Oruvail)
  Formulations: Injection 100mg, | Patch 30mg, 

## Assembly — Collect All Chunks
### Cell 24 — Assemble all 32 chunks

In [38]:
# ── CELL 24 · Assemble all chunks ─────────────────────────────────────────────

chunks: list[dict] = []

# ── 1. Ch1 — Skin Anatomy ─────────────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "Chapter 1 — Skin Anatomy & Wound Relevance",
    parent_section = "Basic Wound Principles",
    text           = ch1_full,
))

# ── 2. Ch2 — Wound Classification ─────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "Chapter 2 — Wound Definition & Classification",
    parent_section = "Basic Wound Principles",
    text           = ch2_full,
))

# ── 3. Ch3 — Wound Assessment ─────────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "Chapter 3 — Wound Assessment Principles",
    parent_section = "Basic Wound Principles",
    text           = ch3_full,
))

# ── 4. Ch4a — Wound Infection & Bacteriology ──────────────────────────────────
chunks.append(make_chunk(
    section        = "Chapter 4 — Wound Infection Pathway & Bacteriology",
    parent_section = "Basic Wound Principles",
    text           = ch4a_full,
))

# ── 5. Ch4b — Antibiotic Treatment ────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "Chapter 4 — Antibiotic Treatment for Wound Infection",
    parent_section = "Basic Wound Principles",
    text           = ch4b_full,
))

# ── 6. Ch7a — Burn Wound ──────────────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "Chapter 7a — Burn Wound: Assessment & Management",
    parent_section = "Management of Acute Wound",
    text           = ch7a_full,
))

# ── 7. Ch7b — Traumatic Wound ─────────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "Chapter 7b — Traumatic Wound: Assessment & Management",
    parent_section = "Management of Acute Wound",
    text           = ch7b_full,
))

# ── 8. Ch8a-A — DFU Assessment & Classification ───────────────────────────────
chunks.append(make_chunk(
    section        = "Chapter 8a — Diabetic Foot Ulcer: Assessment & Wagner Classification",
    parent_section = "Management of Chronic Wound",
    text           = ch8a_A_full,
))

# ── 9. Ch8a-B — DFU Management & Foot Care ────────────────────────────────────
chunks.append(make_chunk(
    section        = "Chapter 8a — Diabetic Foot Ulcer: Management & Foot Care",
    parent_section = "Management of Chronic Wound",
    text           = ch8a_B_full,
))

# ── 10. Ch8b — Venous Ulcer ───────────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "Chapter 8b — Venous Ulcer: Classification, Risk & Treatment",
    parent_section = "Management of Chronic Wound",
    text           = ch8b_full,
))

# ── 11. Ch8c — Arterial Ulcer ─────────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "Chapter 8c — Arterial Ulcer: Diagnosis & Management",
    parent_section = "Management of Chronic Wound",
    text           = ch8c_full,
))

# ── 12. Ch8d — Pressure Ulcer ─────────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "Chapter 8d — Pressure Ulcer: Pathophysiology, Staging & Management",
    parent_section = "Management of Chronic Wound",
    text           = ch8d_full,
))

# ── 13. Ch9 — Non-Healing Ulcer ───────────────────────────────────────────────
for i, part in enumerate(split_long_chunk(ch9_full)):
    chunks.append(make_chunk(
        section        = "Chapter 9 — Non-Healing Ulcer: Causes & Management",
        parent_section = "Management of Chronic Wound",
        text           = part,
        chunk_index    = i,
    ))

# ── 14. Ch10 — Life-Threatening Wounds ────────────────────────────────────────
for i, part in enumerate(split_long_chunk(ch10_full)):
    chunks.append(make_chunk(
        section        = "Chapter 10 — Life-Threatening Wounds (Necrotizing Fasciitis)",
        parent_section = "Management of Chronic Wound",
        text           = part,
        chunk_index    = i,
    ))

# ── 15. Ch11 — Analgesia ──────────────────────────────────────────────────────
for i, part in enumerate(ch11_parts):
    chunks.append(make_chunk(
        section        = "Chapter 11 — Pain Management in Wound Dressing Procedures",
        parent_section = "Practical Aspects in Wound Care",
        text           = part,
        chunk_index    = i,
    ))

# ── 16. Ch13 — Wound Cleansing ────────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "Chapter 13 — Wound Cleansing Solutions",
    parent_section = "Practical Aspects in Wound Care",
    text           = ch13_full,
))

# ── 17. Ch14 — Dressing Overview ──────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "Chapter 14 — Dressing Purpose & Categories Overview",
    parent_section = "Types of Dressing",
    text           = ch14_overview_full,
))

# ── 18–27. Ch14 — One chunk per modern dressing ───────────────────────────────
for name, text in dressing_chunks.items():
    if len(text) >= MIN_CHUNK_CHARS:
        chunks.append(make_chunk(
            section        = f"Chapter 14 — Modern Dressing: {name}",
            parent_section = "Types of Dressing",
            text           = text,
        ))

# ── 28. Ch15 — Debridement ────────────────────────────────────────────────────
for i, part in enumerate(ch15_parts):
    chunks.append(make_chunk(
        section        = "Chapter 15 — Wound Debridement Methods",
        parent_section = "Practical Aspects in Wound Care",
        text           = part,
        chunk_index    = i,
    ))

# ── 29. Ch16a — Honey Dressing ────────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "Chapter 16a — Honey Dressing: Principles & Application",
    parent_section = "Adjunctive Treatment",
    text           = ch16a_full,
))

# ── 30. Ch16c — NPWT ──────────────────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "Chapter 16c — NPWT: Mechanism, Indications & Settings",
    parent_section = "Adjunctive Treatment",
    text           = ch16c_full,
))

# ── 31. Appendix 7 — Analgesics ───────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "Appendix 7 — Analgesics Formulations & Dosage",
    parent_section = "Reference",
    text           = app7_text,
))

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Total chunks assembled: {len(chunks)}")
print()
for c in chunks:
    print(f"  [{c['chunk_index']}] {c['section']!r:70s}  chars={c['char_count']:5d}")


Total chunks assembled: 40

  [0] 'Chapter 1 — Skin Anatomy & Wound Relevance'                            chars= 3140
  [0] 'Chapter 2 — Wound Definition & Classification'                         chars= 5281
  [0] 'Chapter 3 — Wound Assessment Principles'                               chars= 2075
  [0] 'Chapter 4 — Wound Infection Pathway & Bacteriology'                    chars= 5025
  [0] 'Chapter 4 — Antibiotic Treatment for Wound Infection'                  chars= 3275
  [0] 'Chapter 7a — Burn Wound: Assessment & Management'                      chars= 8673
  [0] 'Chapter 7b — Traumatic Wound: Assessment & Management'                 chars= 5705
  [0] 'Chapter 8a — Diabetic Foot Ulcer: Assessment & Wagner Classification'  chars= 4564
  [0] 'Chapter 8a — Diabetic Foot Ulcer: Management & Foot Care'              chars= 3284
  [0] 'Chapter 8b — Venous Ulcer: Classification, Risk & Treatment'           chars= 4724
  [0] 'Chapter 8c — Arterial Ulcer: Diagnosis & Management'             

## Quality Validation
### Cell 25 — Content & coverage checks

In [39]:
# ── CELL 25 · Quality validation ──────────────────────────────────────────────

all_text = ' '.join(c['text'].lower() for c in chunks)

MUST_CONTAIN = [
    # Ch1 anatomy
    ('epidermis',              'Skin anatomy — epidermis'),
    ('dermis',                 'Skin anatomy — dermis'),
    ('keratinocyte',           'Skin anatomy — keratinocyte'),
    # Ch2 classification & healing
    ('wound healing',          'Wound healing definition'),
    ('haemostasis',            'Healing phase — haemostasis'),
    ('granulation',            'Healing phase — granulation'),
    ('proliferation',          'Healing phase — proliferation'),
    # Ch4 infection
    ('antibiotic',             'Ch4 — antibiotic treatment'),
    ('c&s',                    'Ch4 — culture & sensitivity'),
    # Ch7a burn
    ('tbsa',                   'Ch7a — TBSA burn assessment'),
    ('lund and browder',       'Ch7a — Lund and Browder chart'),
    ('referral',               'Ch7a/8 — referral criteria'),
    # Ch8 chronic wounds
    ('wagner',                 'Ch8a — Wagner classification'),
    ('diabetic foot',          'Ch8a — diabetic foot'),
    ('venous ulcer',           'Ch8b — venous ulcer'),
    ('compression',            'Ch8b — compression therapy'),
    ('absi',                   'Ch8c — ABSI / ankle-brachial'),
    ('arterial',               'Ch8c — arterial ulcer'),
    ('stage i',                'Ch8d — pressure ulcer staging'),
    ('stage iv',               'Ch8d — stage IV'),
    # Ch9 & Ch10
    ('non-healing',            'Ch9 — non-healing ulcer'),
    ('necrotizing fasciitis',  'Ch10 — necrotizing fasciitis'),
    # Ch13 cleansing
    ('chlorhexidine',          'Ch13 — chlorhexidine cleansing'),
    # Ch14 dressings
    ('film',                   'Ch14 — Film dressing'),
    ('hydrogel',               'Ch14 — Hydrogel dressing'),
    ('hydrocolloid',           'Ch14 — Hydrocolloid dressing'),
    ('alginate',               'Ch14 — Alginate dressing'),
    ('foam',                   'Ch14 — Foam dressing'),
    ('hydrofibre',             'Ch14 — Hydrofibre dressing'),
    ('charcoal',               'Ch14 — Charcoal dressing'),
    ('silver',                 'Ch14 — Silver dressing'),
    ('polymeric membrane',     'Ch14 — Multi-function/Polymeric'),
    ('composite',              'Ch14 — Composite dressing'),
    # Ch15 debridement
    ('debridement',            'Ch15 — debridement'),
    ('autolytic',              'Ch15 — autolytic debridement'),
    ('maggot',                 'Ch15 — biological/maggot debridement'),
    # Ch16a honey
    ('honey',                  'Ch16a — honey dressing'),
    # Ch16c NPWT
    ('npwt',                   'Ch16c — NPWT'),
    ('negative pressure',      'Ch16c — negative pressure'),
    ('125 mmhg',               'Ch16c — NPWT recommended settings'),
    # Appendix 7
    ('paracetamol',            'App7 — paracetamol'),
    ('morphine',               'App7/Ch11 — morphine'),
]

print('Content coverage check:')
all_pass = True
for keyword, label in MUST_CONTAIN:
    found = keyword.lower() in all_text
    status = '✅' if found else '❌ MISSING'
    if not found:
        all_pass = False
    print(f'  {status}  {label}')

print()
# Minimum chunk length
short = [c for c in chunks if c['char_count'] < MIN_CHUNK_CHARS]
print(f'Chunks below MIN_CHUNK_CHARS ({MIN_CHUNK_CHARS}): {len(short)}')
for c in short:
    print(f'  ❌ {c["section"]!r}  chars={c["char_count"]}')

# Duplicate chunk IDs
ids = [c['chunk_id'] for c in chunks]
dupes = [x for x in ids if ids.count(x) > 1]
print(f'Duplicate chunk IDs: {len(set(dupes))}')

print()
print(f'Total chunks: {len(chunks)}')
char_counts = [c["char_count"] for c in chunks]
print(f'Char count — min={min(char_counts)}, max={max(char_counts)}, mean={sum(char_counts)//len(char_counts)}')

if all_pass:
    print('\n✅  All content checks PASSED')
else:
    print('\n⚠️  Some content checks FAILED — review missing sections above')


Content coverage check:
  ✅  Skin anatomy — epidermis
  ✅  Skin anatomy — dermis
  ❌ MISSING  Skin anatomy — keratinocyte
  ✅  Wound healing definition
  ✅  Healing phase — haemostasis
  ✅  Healing phase — granulation
  ✅  Healing phase — proliferation
  ✅  Ch4 — antibiotic treatment
  ✅  Ch4 — culture & sensitivity
  ✅  Ch7a — TBSA burn assessment
  ✅  Ch7a — Lund and Browder chart
  ✅  Ch7a/8 — referral criteria
  ✅  Ch8a — Wagner classification
  ✅  Ch8a — diabetic foot
  ✅  Ch8b — venous ulcer
  ✅  Ch8b — compression therapy
  ✅  Ch8c — ABSI / ankle-brachial
  ✅  Ch8c — arterial ulcer
  ✅  Ch8d — pressure ulcer staging
  ✅  Ch8d — stage IV
  ❌ MISSING  Ch9 — non-healing ulcer
  ✅  Ch10 — necrotizing fasciitis
  ✅  Ch13 — chlorhexidine cleansing
  ✅  Ch14 — Film dressing
  ✅  Ch14 — Hydrogel dressing
  ✅  Ch14 — Hydrocolloid dressing
  ✅  Ch14 — Alginate dressing
  ✅  Ch14 — Foam dressing
  ✅  Ch14 — Hydrofibre dressing
  ✅  Ch14 — Charcoal dressing
  ✅  Ch14 — Silver dressing
  ✅  

In [40]:
len(chunks)

40

In [42]:
chunks[0]

{'chunk_id': '0305af9f4828',
 'source': 'Wound Care Manual - First Edition.pdf',
 'section': 'Chapter 1 — Skin Anatomy & Wound Relevance',
 'parent_section': 'Basic Wound Principles',
 'chunk_index': 0,
 'char_count': 3140,
 'text': "1 Clinical Applied Anatomy in\nWound Care\n\n1‐1. What is the Skin?\n\x83 \nSkin is the outer covering of the body and thus provides protection.\n\x83 \nIt is the largest organ in our body in term of weight and surface areas.\n\x83 \nIts thickness ranges from 0.5 to 4.0 mm depending on location.\n\x83 \nIt consists of different tissues that are joined together to perform several \nessential functions.\n\x83 \nIt is a dynamic organ in a constant of change; whereby the outer layers \nare  continuously  shed  and  replaced  by  the  inner  cells  moving  to  the \nsurface.\n\x83 \n Structurally, the skin consists of 3 principal layers.\n• \nEpidermis:  outer  most  layer,  thinner  portion,  composed  of \nepithelium.\n• \nDermis: middle layer, thicker, consi

In [43]:
# ── CELL 13 · LLM ai_summary (optional) ───────────────────────────────────────
ENABLE_AI_SUMMARY = True   # ← set True when OpenAI key is available

if ENABLE_AI_SUMMARY:
    import os
    from openai import OpenAI
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    SYSTEM_PROMPT = (
        "You are a medical summarisation assistant. "
        "Rewrite the following wound-care guideline text as a clear, complete, self-contained "
        "clinical summary suitable for retrieval-augmented generation. "
        "Preserve all clinical facts, dressing names, wound types, indications, and "
        "contraindications. Return only the summary text — no preamble."
    )

    print("Running AI summaries...")
    for i, c in enumerate(chunks):
        print(f"  [{i+1}/{len(chunks)}] {c['section']}")
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": c["text"]},
            ],
        )
        c["ai_summary"] = resp.choices[0].message.content.strip()
    print("✅ AI summaries done")
else:
    print("ℹ️  AI summary disabled — ai_summary == text (raw chunk)")

Running AI summaries...
  [1/40] Chapter 1 — Skin Anatomy & Wound Relevance
  [2/40] Chapter 2 — Wound Definition & Classification
  [3/40] Chapter 3 — Wound Assessment Principles
  [4/40] Chapter 4 — Wound Infection Pathway & Bacteriology
  [5/40] Chapter 4 — Antibiotic Treatment for Wound Infection
  [6/40] Chapter 7a — Burn Wound: Assessment & Management
  [7/40] Chapter 7b — Traumatic Wound: Assessment & Management
  [8/40] Chapter 8a — Diabetic Foot Ulcer: Assessment & Wagner Classification
  [9/40] Chapter 8a — Diabetic Foot Ulcer: Management & Foot Care
  [10/40] Chapter 8b — Venous Ulcer: Classification, Risk & Treatment
  [11/40] Chapter 8c — Arterial Ulcer: Diagnosis & Management
  [12/40] Chapter 8d — Pressure Ulcer: Pathophysiology, Staging & Management
  [13/40] Chapter 9 — Non-Healing Ulcer: Causes & Management
  [14/40] Chapter 9 — Non-Healing Ulcer: Causes & Management
  [15/40] Chapter 10 — Life-Threatening Wounds (Necrotizing Fasciitis)
  [16/40] Chapter 10 — Life-Thr

## Export
### Cell 26 — Export to JSON

In [44]:
# ── CELL 26 · Export to JSON ──────────────────────────────────────────────────

out_path = OUT_DIR / "WCM_wound_care_manual_kept.json"
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"✅  Exported {len(chunks)} chunks → {out_path}")
print(f"   File size: {out_path.stat().st_size:,} bytes")

# Quick preview of first 3 and last chunk
for c in chunks[:3] + [chunks[-1]]:
    print(f"\n  chunk_id={c['chunk_id']}  section={c['section']!r}  chars={c['char_count']}")
    print(f"  text preview: {c['text'][:120]!r}...")

doc.close()
print("\n✅  PDF closed — ingestion complete")


✅  Exported 40 chunks → ..\ingestion_output_ai\WCM_wound_care_manual_kept.json
   File size: 212,652 bytes

  chunk_id=0305af9f4828  section='Chapter 1 — Skin Anatomy & Wound Relevance'  chars=3140
  text preview: '1 Clinical Applied Anatomy in\nWound Care\n\n1‐1. What is the Skin?\n\x83 \nSkin is the outer covering of the body and thus prov'...

  chunk_id=06f9b3a7dc69  section='Chapter 2 — Wound Definition & Classification'  chars=5281
  text preview: 'Definition and Classification of Wound, \nand Stages of Wound Healing\nDr Haris Ali Chemok Ali/ Dr Mohammad Anwar Hau Abdu'...

  chunk_id=9f8aabde769e  section='Chapter 3 — Wound Assessment Principles'  chars=2075
  text preview: 'Wound Assessment and \nDocumentation\n\n3‐1. Wound Assessment\nGeneral assessment: \nThe general assessment is to identify an'...

  chunk_id=c799dd10dcf3  section='Appendix 7 — Analgesics Formulations & Dosage'  chars=2646
  text preview: 'Appendix 7 — Formulations and Dosage of Commonly Used Analgesics\n(S